# Java Practical Problem Generator + Code Evaluator

Use this notebook to:
1. Generate a random **one-line Java problem statement** from your syllabus topics.
2. Paste/type your Java code in an input box.
3. Compile + run it (when possible) and get a correctness report.


In [ ]:

import os
import re
import shutil
import subprocess
import tempfile
from dataclasses import dataclass
from random import choice

try:
    import ipywidgets as widgets
    from IPython.display import display, Markdown
except Exception as e:
    raise RuntimeError("This notebook needs ipywidgets. In Colab run: !pip -q install ipywidgets") from e

TOPICS = [
    {
        "id": 1,
        "title": "Data types, variables, operators, arrays, control structures",
        "base": "Write a Java program that demonstrates data types, variables, operators, arrays, and control structures.",
        "checks": [r"\b(int|double|float|char|boolean|long|short|byte|String)\b", r"\b(if|switch|for|while|do)\b", r"\[\]"]
    },
    {
        "id": 2,
        "title": "Class and constructors with constructor overloading",
        "base": "Develop a Java program defining a class with multiple constructors to demonstrate constructor overloading.",
        "checks": [r"\bclass\b", r"\b[A-Za-z_]\w*\s*\(", r"\bthis\s*\("]
    },
    {
        "id": 3,
        "title": "Inheritance and method overriding",
        "base": "Develop a Java program showing inheritance and method overriding.",
        "checks": [r"\bextends\b", r"@Override"]
    },
    {
        "id": 4,
        "title": "Exception handling",
        "base": "Develop a Java program to demonstrate exception handling using try-catch-finally.",
        "checks": [r"\btry\b", r"\bcatch\b"]
    },
    {
        "id": 5,
        "title": "Multithreading",
        "base": "Develop a Java program to demonstrate multithreading.",
        "checks": [r"\bThread\b|\bRunnable\b", r"\bstart\s*\("]
    },
    {
        "id": 6,
        "title": "I/O operations",
        "base": "Develop a Java program to demonstrate file I/O operations.",
        "checks": [r"java\.io|java\.nio", r"\b(File|BufferedReader|BufferedWriter|Scanner|Files)\b"]
    },
    {
        "id": 7,
        "title": "Database handling",
        "base": "Develop a Java program to demonstrate database handling (JDBC).",
        "checks": [r"java\.sql", r"\b(Connection|PreparedStatement|Statement|ResultSet|DriverManager)\b"]
    },
    {
        "id": 8,
        "title": "Network programming",
        "base": "Develop a Java program to demonstrate network programming.",
        "checks": [r"java\.net", r"\b(Socket|ServerSocket|URL|HttpURLConnection)\b"]
    },
    {
        "id": 9,
        "title": "Applet structure and event handling",
        "base": "Develop a Java program to demonstrate applet structure and event handling.",
        "checks": [r"\bApplet\b|JApplet", r"java\.awt\.event|ActionListener|MouseListener"]
    },
    {
        "id": 10,
        "title": "Layout managers",
        "base": "Develop a Java program to demonstrate Java layout managers.",
        "checks": [r"Layout", r"\b(BorderLayout|FlowLayout|GridLayout|BoxLayout|CardLayout)\b"]
    },
    {
        "id": 11,
        "title": "Mini project",
        "base": "Build a small Java mini-project integrating OOP, error handling, and user interaction.",
        "checks": [r"\bclass\b", r"\bpublic\s+static\s+void\s+main\s*\("]
    },
]

ONE_LINE_PATTERNS = [
    "Create a concise Java solution for: {title}.",
    "Write a one-file Java program that demonstrates {title}.",
    "Implement a Java example focused on {title}.",
    "Build a Java demo that clearly shows {title}.",
]

@dataclass
class EvalResult:
    compiled: bool
    ran: bool
    required_checks_passed: bool
    missing_requirements: list
    stdout: str
    stderr: str
    score: int

current_topic = None
current_problem = None


def generate_problem(_=None):
    global current_topic, current_problem
    current_topic = choice(TOPICS)
    statement = choice(ONE_LINE_PATTERNS).format(title=current_topic["title"].lower())
    current_problem = statement
    problem_out.value = f"**Practical {current_topic['id']}**: {statement}"


def _extract_public_class_name(code: str):
    m = re.search(r"public\s+class\s+([A-Za-z_]\w*)", code)
    return m.group(1) if m else None


def _requirement_scan(code: str, patterns):
    missing = []
    for p in patterns:
        if not re.search(p, code, flags=re.IGNORECASE | re.MULTILINE):
            missing.append(p)
    return missing


def evaluate_java(code: str) -> EvalResult:
    if not current_topic:
        return EvalResult(False, False, False, ["Generate a problem first."], "", "", 0)

    if not code.strip():
        return EvalResult(False, False, False, ["Java code is empty."], "", "", 0)

    javac = shutil.which("javac")
    java = shutil.which("java")
    if not javac or not java:
        missing = []
        if not javac: missing.append("javac")
        if not java: missing.append("java")
        req_missing = _requirement_scan(code, current_topic["checks"])
        return EvalResult(
            compiled=False,
            ran=False,
            required_checks_passed=(len(req_missing) == 0),
            missing_requirements=req_missing + [f"Runtime tool missing: {', '.join(missing)}"],
            stdout="",
            stderr="Java compiler/runtime not found in this notebook environment.",
            score=max(0, 40 - 10*len(req_missing))
        )

    with tempfile.TemporaryDirectory() as td:
        class_name = _extract_public_class_name(code) or "Main"
        java_file = os.path.join(td, f"{class_name}.java")

        # If there's no public class, compile from Main.java by wrapping is unsafe; we rely on provided code.
        if _extract_public_class_name(code) is None:
            java_file = os.path.join(td, "Main.java")

        with open(java_file, "w", encoding="utf-8") as f:
            f.write(code)

        compile_proc = subprocess.run([javac, java_file], capture_output=True, text=True)
        compiled = compile_proc.returncode == 0

        run_stdout, run_stderr, ran = "", "", False
        if compiled:
            run_class = _extract_public_class_name(code) or "Main"
            run_proc = subprocess.run([java, "-cp", td, run_class], capture_output=True, text=True)
            ran = run_proc.returncode == 0
            run_stdout, run_stderr = run_proc.stdout, run_proc.stderr

    missing_reqs = _requirement_scan(code, current_topic["checks"])
    req_ok = len(missing_reqs) == 0

    score = 0
    score += 40 if compiled else 0
    score += 30 if ran else 0
    score += 30 if req_ok else max(0, 30 - len(missing_reqs) * 10)

    stderr_all = (compile_proc.stderr if not compiled else run_stderr).strip()

    return EvalResult(compiled, ran, req_ok, missing_reqs, run_stdout.strip(), stderr_all, score)


def on_evaluate(_=None):
    result = evaluate_java(code_input.value)

    status = []
    status.append(f"- **Compiled:** {'✅' if result.compiled else '❌'}")
    status.append(f"- **Ran successfully:** {'✅' if result.ran else '❌'}")
    status.append(f"- **Topic requirements matched:** {'✅' if result.required_checks_passed else '❌'}")
    status.append(f"- **Score (heuristic):** **{result.score}/100**")

    if result.missing_requirements:
        status.append("- **Missing requirement patterns:**")
        status.extend([f"  - `{m}`" for m in result.missing_requirements])

    out = "
".join(status)

    if result.stdout:
        out += f"

### Program Output
```
{result.stdout}
```"
    if result.stderr:
        out += f"

### Errors / Warnings
```
{result.stderr}
```"

    out_md.value = out


gen_btn = widgets.Button(description="Generate Random Problem", button_style="info")
eval_btn = widgets.Button(description="Evaluate Java Code", button_style="success")
problem_out = widgets.HTML(value="Click **Generate Random Problem** to start.")
code_input = widgets.Textarea(
    value="public class Main {
    public static void main(String[] args) {
        System.out.println("Hello Java");
    }
}",
    placeholder="Paste your Java code here...",
    description="Code:",
    layout=widgets.Layout(width="100%", height="320px"),
)
out_md = widgets.HTML(value="")

gen_btn.on_click(generate_problem)
eval_btn.on_click(on_evaluate)

display(widgets.VBox([
    widgets.HTML("<h3>Java Practical Generator</h3>"),
    gen_btn,
    problem_out,
    widgets.HTML("<h3>Paste / Type Java Code</h3>"),
    code_input,
    eval_btn,
    widgets.HTML("<h3>Evaluation Result</h3>"),
    out_md,
]))


## Notes
- The checker uses a **heuristic** score (compile + run + topic pattern checks).
- For some topics (DB, applet, networking), runtime may fail unless dependencies/environment are available.
- If Java is not installed in the notebook runtime, install JDK first (for Colab: `!apt-get -qq install openjdk-17-jdk`).
